# 1. Install & Import

In [ ]:
!pip install datasets sacrebleu -q

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

import numpy as np
import re, random, time, math
from collections import Counter
import sacrebleu
from datasets import load_dataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

## 2. Load Dataset — Tatoeba (ar → en)

> **Why Tatoeba?**  
> opus_books is literary text — long, complex, varied. Tatoeba contains short everyday sentences
> ("Where is the station?", "I like coffee") that are much easier for an LSTM to learn from.
> This alone will significantly reduce your loss.

In [ ]:
from datasets import load_dataset

# Fallback: opus100 ar-en (standard Parquet format, no script needed)
dataset = load_dataset('Helsinki-NLP/opus-100', 'ar-en')
print(dataset)

all_data = []
for split in dataset:
    for ex in dataset[split]:
        all_data.append((ex['translation']['ar'], ex['translation']['en']))

print(f'\nTotal pairs: {len(all_data)}')
print('Sample:', all_data[0])

In [ ]:
# Keep only 80k pairs — enough to learn, fast enough to train
random.shuffle(all_data)
all_data = all_data[:80_000]
print(f'Using: {len(all_data)} pairs')


## 3. Preprocessing

In [ ]:
def normalize_arabic(text):
    text = re.sub(r'[\u0610-\u061A\u064B-\u065F]', '', text)   # remove diacritics
    text = re.sub(r'[\u0622\u0623\u0625\u0671]', '\u0627', text) # normalize alef
    text = re.sub(r'\u0629', '\u0647', text)                     # ta marbuta → ha
    text = re.sub(r'\u0649', '\u064A', text)                     # alef maqsura → ya
    text = re.sub(r'\u0640', '', text)                           # remove tatweel
    text = re.sub(r'[^\u0600-\u06FF\s]', ' ', text)             # keep Arabic only
    return re.sub(r'\s+', ' ', text).strip()

def normalize_english(text):
    text = text.lower()
    text = re.sub(r"[^a-z\s']", ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

def tokenize(text):
    return text.split()

# --- Filter: keep only short, clean pairs ---
MAX_LEN   = 20   # tighter than v1 (was 30) → cleaner pairs
MIN_LEN   = 2

pairs = []
for ar_raw, en_raw in all_data:
    ar_tok = tokenize(normalize_arabic(ar_raw))
    en_tok = tokenize(normalize_english(en_raw))
    if MIN_LEN <= len(ar_tok) <= MAX_LEN and MIN_LEN <= len(en_tok) <= MAX_LEN:
        pairs.append((ar_tok, en_tok))

random.shuffle(pairs)
print(f'Usable pairs after filtering: {len(pairs)}')

# Print length distribution
ar_lens = [len(p[0]) for p in pairs]
en_lens = [len(p[1]) for p in pairs]
print(f'AR avg length: {np.mean(ar_lens):.1f} | EN avg length: {np.mean(en_lens):.1f}')

# Show some examples
for ar, en in pairs[:5]:
    print(f'  AR: {" ".join(ar)}')
    print(f'  EN: {" ".join(en)}')
    print()

## 4. Vocabulary

In [ ]:
PAD_IDX, SOS_IDX, EOS_IDX, UNK_IDX = 0, 1, 2, 3
SPECIAL = ['<pad>', '<sos>', '<eos>', '<unk>']

class Vocabulary:
    def __init__(self, min_freq=2):
        self.min_freq  = min_freq
        self.word2idx  = {t: i for i, t in enumerate(SPECIAL)}
        self.idx2word  = {i: t for t, i in self.word2idx.items()}

    def build(self, token_lists):
        counter = Counter(tok for toks in token_lists for tok in toks)
        for word, freq in sorted(counter.items(), key=lambda x: -x[1]):
            if freq >= self.min_freq and word not in self.word2idx:
                idx = len(self.word2idx)
                self.word2idx[word] = idx
                self.idx2word[idx]  = word

    def encode(self, tokens):
        return [self.word2idx.get(t, UNK_IDX) for t in tokens]

    def decode(self, indices):
        words = []
        for i in indices:
            w = self.idx2word.get(i, '<unk>')
            if w in ('<eos>', '<pad>'): break
            if w not in ('<sos>',): words.append(w)
        return words

    def __len__(self): return len(self.word2idx)

ar_vocab = Vocabulary(min_freq=2)
en_vocab = Vocabulary(min_freq=2)

ar_vocab.build([p[0] for p in pairs])
en_vocab.build([p[1] for p in pairs])

print(f'Arabic  vocab: {len(ar_vocab):,}')
print(f'English vocab: {len(en_vocab):,}')

## 5. Dataset & DataLoader

In [ ]:
class TranslationDataset(Dataset):
    def __init__(self, pairs, src_vocab, tgt_vocab):
        self.data = []
        for src_toks, tgt_toks in pairs:
            src_ids = src_vocab.encode(src_toks)
            tgt_ids = [SOS_IDX] + tgt_vocab.encode(tgt_toks) + [EOS_IDX]
            self.data.append((src_ids, tgt_ids))

    def __len__(self): return len(self.data)
    def __getitem__(self, idx): return self.data[idx]


def collate_fn(batch):
    src_batch, tgt_batch = zip(*batch)
    src_lens = torch.tensor([len(s) for s in src_batch])
    tgt_lens = torch.tensor([len(t) for t in tgt_batch])
    max_src  = src_lens.max().item()
    max_tgt  = tgt_lens.max().item()

    src_padded = torch.full((len(src_batch), max_src), PAD_IDX, dtype=torch.long)
    tgt_padded = torch.full((len(tgt_batch), max_tgt), PAD_IDX, dtype=torch.long)

    for i, (s, t) in enumerate(zip(src_batch, tgt_batch)):
        src_padded[i, :len(s)] = torch.tensor(s)
        tgt_padded[i, :len(t)] = torch.tensor(t)

    return src_padded, tgt_padded, src_lens, tgt_lens


n = len(pairs)
train_pairs = pairs[:int(0.80 * n)]
val_pairs   = pairs[int(0.80 * n):int(0.90 * n)]
test_pairs  = pairs[int(0.90 * n):]
print(f'Train: {len(train_pairs):,} | Val: {len(val_pairs):,} | Test: {len(test_pairs):,}')

BATCH_SIZE = 128   # larger batch → more stable gradients

train_ds = TranslationDataset(train_pairs, ar_vocab, en_vocab)
val_ds   = TranslationDataset(val_pairs,   ar_vocab, en_vocab)
test_ds  = TranslationDataset(test_pairs,  ar_vocab, en_vocab)

train_loader = DataLoader(train_ds, BATCH_SIZE, shuffle=True,  collate_fn=collate_fn, pin_memory=True)
val_loader   = DataLoader(val_ds,   BATCH_SIZE, shuffle=False, collate_fn=collate_fn, pin_memory=True)
test_loader  = DataLoader(test_ds,  BATCH_SIZE, shuffle=False, collate_fn=collate_fn, pin_memory=True)

## 6. Model

### Key design decisions

| Component | v1 | v2 | Why |
|---|---|---|---|
| Encoder | Unidirectional LSTM | **Bidirectional** LSTM | Sees full context before generating |
| Hidden state handoff | Direct copy | **Linear projection** | Bidir has 2×H, decoder needs H |
| Attention score | `h_dec · h_enc` | `(h_dec · h_enc) / √H` | Scaled → avoids vanishing softmax gradients |
| Loss | CrossEntropy | **CrossEntropy + label smoothing** | Less overconfident, better generalisation |

In [ ]:
# ─── Encoder (Bidirectional) ───────────────────────────────────────────────────
class Encoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, n_layers, dropout):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.n_layers   = n_layers
        self.embedding  = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)
        self.lstm = nn.LSTM(
            embed_dim, hidden_dim, n_layers,
            batch_first=True,
            dropout=dropout if n_layers > 1 else 0,
            bidirectional=True           # <── NEW
        )
        # Project 2*hidden → hidden so decoder can use it directly
        self.fc_h = nn.Linear(hidden_dim * 2, hidden_dim)
        self.fc_c = nn.Linear(hidden_dim * 2, hidden_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, src, src_lens):
        embedded = self.dropout(self.embedding(src))           # (B, T, E)
        packed   = pack_padded_sequence(embedded, src_lens.cpu(),
                                        batch_first=True, enforce_sorted=False)
        enc_out_packed, (hidden, cell) = self.lstm(packed)
        enc_out, _ = pad_packed_sequence(enc_out_packed, batch_first=True)
        # enc_out: (B, T, 2*H)

        # hidden/cell: (n_layers*2, B, H) → merge directions per layer
        # Reshape: (n_layers, 2, B, H) → cat on dim 2 → (n_layers, B, 2H) → project
        hidden = hidden.view(self.n_layers, 2, -1, self.hidden_dim)
        cell   = cell.view(self.n_layers,   2, -1, self.hidden_dim)
        hidden = torch.tanh(self.fc_h(torch.cat([hidden[:, 0], hidden[:, 1]], dim=-1)))
        cell   = torch.tanh(self.fc_c(torch.cat([cell[:, 0],   cell[:, 1]],   dim=-1)))
        # hidden/cell now: (n_layers, B, H)

        return enc_out, hidden, cell   # enc_out still has 2*H channels


# ─── Scaled Dot-Product Attention ─────────────────────────────────────────────
class DotProductAttention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        # project decoder hidden (H) → encoder output dim (2H) for dot product
        self.proj = nn.Linear(hidden_dim, hidden_dim * 2, bias=False)

    def forward(self, dec_hidden, enc_out, src_mask=None):
        # dec_hidden : (B, 1, H)
        # enc_out    : (B, T, 2H)
        H2 = enc_out.size(-1)
        dec_proj = self.proj(dec_hidden)                               # (B, 1, 2H)
        scores   = torch.bmm(dec_proj, enc_out.transpose(1, 2))       # (B, 1, T)
        scores   = scores / math.sqrt(H2)
        if src_mask is not None:
            scores = scores.masked_fill(src_mask.unsqueeze(1), float('-inf'))
        weights  = F.softmax(scores, dim=-1)                          # (B, 1, T)
        context  = torch.bmm(weights, enc_out)                        # (B, 1, 2H)
        return context, weights.squeeze(1)


# ─── Decoder ──────────────────────────────────────────────────────────────────
class Decoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, n_layers, dropout):
        super().__init__()
        enc_out_dim = hidden_dim * 2    # bidirectional encoder
        self.embedding  = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)
        self.attention = DotProductAttention(hidden_dim)
        self.lstm = nn.LSTM(
            embed_dim + enc_out_dim, hidden_dim, n_layers,
            batch_first=True,
            dropout=dropout if n_layers > 1 else 0
        )
        self.fc_out  = nn.Linear(hidden_dim + enc_out_dim, vocab_size)
        self.dropout = nn.Dropout(dropout)

    def forward_step(self, tok, hidden, cell, enc_out, src_mask=None):
        embedded = self.dropout(self.embedding(tok.unsqueeze(1)))     # (B,1,E)
        dec_top  = hidden[-1].unsqueeze(1)                           # (B,1,H)
        context, attn_w = self.attention(dec_top, enc_out, src_mask) # (B,1,2H)
        lstm_in  = torch.cat([embedded, context], dim=-1)            # (B,1,E+2H)
        dec_out, (hidden, cell) = self.lstm(lstm_in, (hidden, cell)) # (B,1,H)
        pred     = self.fc_out(torch.cat([dec_out, context], dim=-1).squeeze(1))  # (B,V)
        return pred, hidden, cell, attn_w

    def forward(self, tgt, hidden, cell, enc_out, src_mask=None, tf_ratio=0.5):
        B, T    = tgt.shape
        outputs = torch.zeros(B, T - 1, self.fc_out.out_features, device=tgt.device)
        tok     = tgt[:, 0]
        for t in range(T - 1):
            pred, hidden, cell, _ = self.forward_step(tok, hidden, cell, enc_out, src_mask)
            outputs[:, t] = pred
            tok = tgt[:, t + 1] if random.random() < tf_ratio else pred.argmax(-1)
        return outputs


# ─── Seq2Seq wrapper ──────────────────────────────────────────────────────────
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def make_src_mask(self, src):
        return (src == PAD_IDX)  # True where padding

    def forward(self, src, src_lens, tgt, tf_ratio=0.5):
        enc_out, hidden, cell = self.encoder(src, src_lens)
        src_mask = self.make_src_mask(src)
        return self.decoder(tgt, hidden, cell, enc_out, src_mask, tf_ratio)


# ─── Instantiate ──────────────────────────────────────────────────────────────
EMBED_DIM  = 256
HIDDEN_DIM = 512
N_LAYERS   = 2
DROPOUT    = 0.3

encoder = Encoder(len(ar_vocab), EMBED_DIM, HIDDEN_DIM, N_LAYERS, DROPOUT)
decoder = Decoder(len(en_vocab), EMBED_DIM, HIDDEN_DIM, N_LAYERS, DROPOUT)
model   = Seq2Seq(encoder, decoder).to(device)

# Weight init: Xavier uniform for linear layers
def init_weights(m):
    if isinstance(m, nn.Linear):
        nn.init.xavier_uniform_(m.weight)
        if m.bias is not None: nn.init.zeros_(m.bias)
    elif isinstance(m, nn.Embedding):
        nn.init.normal_(m.weight, mean=0, std=0.01)
        m.weight.data[PAD_IDX].zero_()

model.apply(init_weights)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model parameters: {total_params:,}')